# DiffusionNFT：原始 NFT 与轨迹 Hybrid-NFT

本笔记严格按以下顺序组织：先定义原始 NFT loss；再区分 Flow Matching 与 Score Matching；然后将两种模型的预测统一转换到 $x_{start}$ 并实现原始 NFT；最后说明轨迹 Hybrid Loss 替换原始 MSE 的位置，并给出 Flow/Score 两个 Hybrid-NFT 实现。

本文代码针对轨迹张量 $(B,T,D)$，其中前两维是 $(x,y)$。图像 latent $(B,C,H,W)$ 没有时间轴，不能直接使用 waypoint 积分项。

# 1. 原始 NFT Loss

设旧策略预测为 $h_{old}$，当前可训练策略为 $h_\theta$，奖励归一化为最优性概率 $r\in[0,1]$。NFT 不训练两个独立网络，而是构造隐式正、负策略：

$$
h_\theta^+=(1-\beta)h_{old}+\beta h_\theta,\qquad h_\theta^-=(1+\beta)h_{old}-\beta h_\theta.
$$

给定干净目标 $x_{start}$、带噪状态 $x_t$ 和模型预测参数化 $h$，原始 NFT 对两个隐式分身分别做回归。本文沿用 HDP 的数据表示：$x_{start}$ 是干净的速度/位移增量轨迹，绝对位置轨迹另记为 $\tau_0^x$：

$$
L_+=\operatorname{MSE}(\hat x_{start}^+,x_{start}),\qquad L_-=\operatorname{MSE}(\hat x_{start}^-,x_{start}).
$$

奖励加权策略损失为：

$$
L_{NFT}=\mathbb E\left[\frac{r}{\beta}L_+ + \frac{1-r}{\beta}L_-\right].
$$

高奖励样本主要优化正分身，低奖励样本主要优化负分身。$h_{ref}$ 的 KL/MSE 正则独立加入：

$$
L_{total}=L_{NFT}+\beta_{KL}\operatorname{MSE}(h_\theta,h_{ref}).
$$

代码中的 $h_{old}$ 和 $h_{ref}$ 必须 stop-gradient 或不参与当前计算图；只有 $h_\theta$ 接收 NFT 策略梯度。

# 2. Flow Matching 与 Score Matching 的区别

NFT 的隐式正负组合、奖励权重和 KL 正则在两种模型中完全相同；必须区分的是模型输出到干净数据 $x_1$ 的转换。Hybrid loss 也必须在这个转换之后计算。

## Flow Matching

Flow Matching 使用条件最优传输（Conditional Optimal Transport, COT）直线路径及其速度场 $u_t$：

$$
x_t=(1-t)x_0+t x_1,\qquad u_t=\frac{d x_t}{dt}=x_1-x_0.
$$

因此：

$$
\hat x_1=x_t+(1-t)\hat u_t.
$$

这里的 $t\in[0,1]$ 是从源分布 $x_0$ 到真实数据 $x_1$ 的路径插值系数；因此 Flow 模型预测的是速度向量场，训练目标中的干净规划轨迹是 $x_1$。

## Score Matching / VP Diffusion

VP 扩散满足：

$$
x_t=\alpha_t x_0+\sigma_t\epsilon.
$$

不同预测头都先还原为干净数据 $x_1$：

$$
\hat x_1^{\epsilon}=\frac{x_t-\sigma_t\hat\epsilon}{\alpha_t},\qquad \hat x_1^{score}=\frac{x_t+\sigma_t^2\hat s}{\alpha_t},
$$
$$
\hat x_1^{v}=\frac{x_t-\sigma_t(\sigma_t x_t+\alpha_t\hat v)}{\alpha_t}.
$$

因此不能把 Flow 的路径时间 $t$ 当成 VP 的噪声标准差 $\sigma_t$，也不能把 flow velocity 当成 diffusion-v。统一原则是：先按各自物理过程转换到 $x_1$，再计算 NFT 的回归损失或轨迹 Hybrid loss。

# 3. Hybrid Loss 替换原始 NFT 的哪一部分？

原始 NFT 的替换边界只有一处：将每个隐式分支在 velocity-space $x_{start}$ 上的 MSE 替换为轨迹 Hybrid loss；正/负分支构造和奖励加权不变。$x_{start}$ 不是绝对位置，而是干净速度/位移增量。

对一个分支，转换得到的 $x_{start}$ 已经是速度/位移增量：

$$
\hat\tau_0^v=\hat x_{start},\qquad\tau_0^v=x_{start}.
$$

Hybrid 分支损失为：

$$
L_{branch}^{hybrid}=\operatorname{MSE}(\hat\tau_0^v,\tau_0^v)+\omega\operatorname{MSE}(M\hat\tau_0^v\Delta t,\tau_0^x).
$$

其中 $M$ 是下三角全 1 积分矩阵。第一项保证局部时间连续性，第二项抵消速度积分造成的全局漂移。于是：

$$
L_{NFT}^{hybrid}=\mathbb E\left[\frac{r}{\beta}L_{pos}^{hybrid}+\frac{1-r}{\beta}L_{neg}^{hybrid}\right].
$$

若令速度误差为 $e$，则单个分支满足：

$$
L_{branch}^{hybrid}=e^T(I+\omega\Delta t^2M^TM)e=\|e\|_P^2.
$$

代码实现先把每个正/负预测转换到 velocity-space $x_{start}$，直接计算第一项，再对 $x_{start}[..., :2]$ 积分计算 waypoint 项。因此 Flow 与 Score 版本的 Hybrid 计算部分完全相同，只有前面的 $x_{start}$ 转换不同。

# 4. 四个实现的调用关系与理论性质

原始 Flow NFT 调用函数 $\texttt{nft\_loss\_original\_flow}$；原始 Score/VP NFT 调用 $\texttt{nft\_loss\_original\_score}$，并通过 $prediction\_type$ 选择 $x_0/\epsilon/score/v$。替换后分别调用 $\texttt{nft\_loss\_hybrid\_flow}$ 与 $\texttt{nft\_loss\_hybrid\_score}$。

完整 Hybrid 积分时，正、负分支只是把欧氏度量 $I$ 换成相同的正定度量：

$$
P=I+\omega\Delta t^2M^TM.
$$

由于 $P$ 与样本奖励、正负分支及待优化输出无关，对 NFT 条件风险求导时，公共可逆矩阵 $P$ 可以从一阶条件中消去。因此完整 Hybrid-NFT 与原始 NFT 的理论最优强化方向相同，但额外强调时间积分后的全局轨迹误差。启用 $detach\_window\_size$ 后，前向 loss 数值不变，反向改为局部窗口代理梯度，此时应将其理解为训练稳定化近似。

若训练数据已归一化，必须传入 $state\_mean=[0,0,0,0]$、$state\_std=[0.5,0.5,1,1]$。直接速度项仍在归一化扩散空间计算；积分前先反归一化为物理增量。

In [ ]:
from typing import Dict, Optional, Tuple
import torch
from torch import Tensor


# -----------------------------------------------------------------------------
# 基础工具函数
# -----------------------------------------------------------------------------

def _batch_scalar_shape(x: Tensor) -> Tuple[int, ...]:
    """
    返回将批标量广播到与 x 兼容的形状。
    例如，如果 x 形状为 (B, T, D)，则返回 (B, 1, 1)。
    这样标量可以与 x 的每个元素相乘或相加。
    """
    return (x.shape[0],) + (1,) * (x.ndim - 1)


def _reward_probability(advantages: Tensor, adv_clip_max: float) -> Tensor:
    """
    将优势值（advantages）映射到 [0,1] 区间，作为最优性概率 r。
    输入 advantages: (B,) 形状的张量。
    输出 r: (B,) 形状的张量，每个值在 [0,1] 内。

    步骤：
    1. 截断优势值到 [-adv_clip_max, adv_clip_max]
    2. 线性映射到 [0,1]：r = (a/adv_clip_max + 1) / 2
    3. 再次截断到 [0,1] 确保数值安全。
    """
    a = advantages.clamp(-adv_clip_max, adv_clip_max)
    return ((a / adv_clip_max) / 2.0 + 0.5).clamp(0.0, 1.0)


# -----------------------------------------------------------------------------
# 加噪过程
# -----------------------------------------------------------------------------

def flow_add_noise(x1: Tensor, t: Tensor, noise: Optional[Tensor] = None) -> Tensor:
    """
    Flow Matching (条件最优传输) 加噪：x_t = (1-t) * x0 + t * x1，其中 x0 ~ N(0,I)。
    输入：
        x1: (B, T, D) 干净数据（也可以是图像等任意形状，但这里针对轨迹）
        t:  (B,) 或标量，时间步
        noise: 可选，形状与 x1 相同，内部采样标准高斯噪声
    输出：
        x_t: (B, T, D) 加噪后的数据
    形状变换说明：
        t 通过 _batch_scalar_shape(x1) 变成 (B, 1, 1)，以便广播到 (B, T, D) 的每个元素。
    """
    if noise is None:
        noise = torch.randn_like(x1)
    t = t.reshape(_batch_scalar_shape(x1))   # (B,) -> (B,1,1) 或标量 -> (1,1,1)
    return (1.0 - t) * noise + t * x1


def vp_add_noise(x1: Tensor, alpha: Tensor, sigma: Tensor, noise: Optional[Tensor] = None) -> Tensor:
    """
    VP-SDE 加噪：x_t = alpha_t * x1 + sigma_t * epsilon，epsilon ~ N(0,I)。
    输入：
        x1: (B, T, D) 干净数据
        alpha: (B,) 或标量
        sigma: (B,) 或标量
        noise: 可选，形状与 x1 相同
    输出：
        x_t: (B, T, D)
    形状变换：
        alpha, sigma 通过 _batch_scalar_shape(x1) 转为 (B,1,1) 以广播。
    """
    if noise is None:
        noise = torch.randn_like(x1)
    alpha = alpha.reshape(_batch_scalar_shape(x1))  # (B,1,1)
    sigma = sigma.reshape(_batch_scalar_shape(x1))  # (B,1,1)
    return alpha * x1 + sigma * noise


# -----------------------------------------------------------------------------
# 模型预测到干净数据 x_start 的转换
# -----------------------------------------------------------------------------

def flow_prediction_to_x1(pred: Tensor, x_t: Tensor, t: Tensor) -> Tensor:
    """
    Flow 模型预测的速度场 v 转换为干净数据 x1。
    公式：x1 = x_t + (1 - t) * v
    输入：
        pred: (B, T, D) 模型预测的速度场
        x_t:  (B, T, D) 加噪数据
        t:    (B,) 或标量
    输出：
        x1: (B, T, D) 预测的干净数据
    形状变换：
        t 重塑为 (B,1,1) 以便广播。
    """
    t = t.reshape(_batch_scalar_shape(pred))   # (B,1,1)
    return x_t + (1.0 - t) * pred


def vp_prediction_to_x1(pred: Tensor, x_t: Tensor, alpha: Tensor, sigma: Tensor,
                        prediction_type: str = 'x0') -> Tensor:
    """
    VP-SDE 各种预测头转换为干净数据 x1。
    支持 prediction_type:
        'x0'    : 直接预测 x1
        'noise' : 预测噪声 eps
        'score' : 预测得分 s = -eps / sigma
        'v'     : 预测 v = alpha * eps - sigma * x1 （标准 VP v-prediction）

    输入：
        pred: (B, T, D) 模型预测
        x_t:  (B, T, D) 加噪数据
        alpha: (B,) 或标量
        sigma: (B,) 或标量
        prediction_type: str
    输出：
        x1: (B, T, D)
    形状变换：
        alpha, sigma 通过 _batch_scalar_shape 转为 (B,1,1) 以广播。
    """
    shape = _batch_scalar_shape(pred)          # (B,1,1)
    alpha = alpha.reshape(shape)
    sigma = sigma.reshape(shape)

    if prediction_type == 'x0':
        return pred
    elif prediction_type == 'noise':
        eps = pred
    elif prediction_type == 'score':
        eps = -sigma * pred
    elif prediction_type == 'v':
        # 由 x_t = alpha * x1 + sigma * eps 和 v = alpha * eps - sigma * x1
        # 解出 eps = sigma * x_t + alpha * v （利用 alpha^2 + sigma^2 = 1）
        eps = sigma * x_t + alpha * pred
    else:
        raise ValueError(f'Unknown VP prediction_type: {prediction_type}')

    # x1 = (x_t - sigma * eps) / alpha
    return (x_t - sigma * eps) / (alpha + 1e-6)


# -----------------------------------------------------------------------------
# 轨迹混合损失 (Hybrid Loss) 的核心部分
# -----------------------------------------------------------------------------

def build_metric_p(
    horizon: int,
    n_integral: int,
    std_integral: Tensor,
    omega: float,
    device: torch.device,
    dtype: torch.dtype,
) -> Tensor:
    """
    构造积分部分各维度的 P 矩阵。
    P_d = I + omega * std_d^2 * M^T M
    其中 M 为下三角积分矩阵 (T,T)，I 为单位阵。

    输入：
        horizon: T（时间长度）
        n_integral: 积分维度数（如 2 代表 x,y）
        std_integral: (n_integral,) 各积分维度的标准差
        omega: 权重
        device, dtype: 设备与数据类型
    输出：
        metric_p: (n_integral, T, T) 每个积分维度一个 P 矩阵
    """
    # 下三角积分矩阵 M (T,T)
    M = torch.tril(torch.ones(horizon, horizon, device=device, dtype=dtype))  # (T,T)
    gram = M.T @ M                                                           # (T,T) = M^T M
    I = torch.eye(horizon, device=device, dtype=dtype)                       # (T,T)
    # std_integral: (n_integral,)，平方后变形为 (n_integral,1,1) 与 gram 相乘
    metric_p = I[None] + omega * std_integral.square()[:, None, None] * gram[None]
    # 返回 (n_integral, T, T)
    return metric_p


def hybrid_loss_per_sample(
    pred_x_start: Tensor,             # (B,T,D) 归一化空间的预测（干净数据）
    target_x_start: Tensor,           # (B,T,D) 归一化空间的目标
    target_waypoints: Tensor,         # (B,T,2) 绝对位置目标，用于截断梯度时
    norm_std: Tensor,                 # (D,) 标准差，用于反归一化积分部分
    n_integral: int,                  # 参与积分的维度数
    omega: float,                     # waypoint 权重
    dt: float,                        # 时间间隔
    detach_window_size: Optional[int],# 截断梯度窗口大小
    adaptive_scale: bool = True,      # 是否使用自适应分母
) -> Tensor:
    """
    计算逐样本的 Hybrid 损失（P 范数形式），返回 (B,)。

    前向使用精确二次型，若 detach_window_size 不为 None 则反向使用窗口截断的代理梯度。

    内部步骤及形状变化：
    1. 计算归一化误差 error (B,T,D)
    2. 拆分积分部分 error_integral (B,T,n_integral) 和非积分部分 error_non_integral (B,T,D-n_integral)
    3. 构造 P 矩阵 metric_p (n_integral,T,T)
    4. 将 error_integral 转置为 (B,n_integral,T) 用于 einsum 计算二次型
    5. 得到积分部分二次型 integral_quad (B,) 和非积分部分 non_integral_mse (B,)
    6. 计算自适应缩放因子 scale (B,) 并缩放得到 loss_forward (B,)
    7. 若截断梯度，构造代理损失：
       - 反归一化积分部分得到物理位移增量 (B,T,n_integral)
       - 带截断的积分得到 integrated (B,T,n_integral)
       - 取前2维与 target_waypoints 比较得到 waypoint_error (B,T,2)
       - 计算代理损失 surrogate (B,)
       - 返回 loss_forward.detach() + surrogate - surrogate.detach()
    """
    B, T, D = pred_x_start.shape
    if n_integral > D:
        raise ValueError("n_integral cannot exceed D")

    # 1. 误差
    error = pred_x_start - target_x_start                     # (B,T,D)

    # 2. 拆分
    error_integral = error[:, :, :n_integral]                # (B,T,n_integral)
    error_non_integral = error[:, :, n_integral:]            # (B,T,D-n_integral)
    std_integral = norm_std[:n_integral]                     # (n_integral,)

    # 3. 构造 P 矩阵
    metric_p = build_metric_p(T, n_integral, std_integral, omega,
                              pred_x_start.device, pred_x_start.dtype)  # (n_integral,T,T)

    # 4. 计算积分部分的二次型 e^T P e
    error_integral_t = error_integral.transpose(1, 2)        # (B,n_integral,T)
    # weighted = P @ e，结果 (B,n_integral,T)
    weighted = torch.einsum('bnt,nst->bns', error_integral_t, metric_p)  # (B,n_integral,T)
    # 逐元素乘积后对 n_integral 和 T 求和，得到每个样本的标量
    integral_quad = (error_integral_t * weighted).sum(dim=(1,2))  # (B,)

    # 非积分部分普通 MSE，对 D-n_integral 和 T 求和
    non_integral_mse = error_non_integral.square().sum(dim=(1,2))     # (B,)

    # 总的二次型（未缩放）
    quadratic = integral_quad + non_integral_mse            # (B,)

    # 5. 自适应分母（基于误差的绝对值均值）
    if adaptive_scale:
        with torch.no_grad():
            # error.abs().mean(dim=(1,2)) 对 T 和 D 求均值，得到 (B,)
            scale = error.abs().mean(dim=(1,2)).clamp_min(1e-5)  # (B,)
    else:
        scale = torch.ones(B, device=pred_x_start.device)

    # 缩放后的前向损失
    loss_forward = quadratic / scale                      # (B,)

    # 如果不截断梯度，直接返回
    if detach_window_size is None:
        return loss_forward

    # ------------------- 截断梯度：构造代理损失 -------------------
    # 反归一化得到物理位移增量（仅积分部分）
    # std_integral 形状 (n_integral,)，广播到 (B,T,n_integral)
    pred_physical_integral = pred_x_start[:, :, :n_integral] * std_integral  # (B,T,n_integral)

    # 带截断的积分（前向数值相同，梯度仅来自最近窗口）
    full_integral = pred_physical_integral.cumsum(dim=1) * dt          # (B,T,n_integral)
    if detach_window_size >= T:
        integrated = full_integral
    else:
        w = detach_window_size
        # recent 部分：当前步 - w 步前的累积和，保留梯度
        recent = torch.cat([full_integral[:, :w],
                            full_integral[:, w:] - full_integral[:, :-w]], dim=1)  # (B,T,n_integral)
        # history 部分：w 步前的累积和，detach 掉梯度
        history = torch.cat([torch.zeros_like(full_integral[:, :w]),
                             full_integral[:, :-w].detach()], dim=1)          # (B,T,n_integral)
        integrated = history + recent                                   # (B,T,n_integral)

    # waypoint 误差（仅前 2 维，因为 target_waypoints 为 (B,T,2)）
    if n_integral < 2:
        raise ValueError("n_integral must be at least 2 to compare waypoints")
    waypoint_error = integrated[:, :, :2] - target_waypoints[:, :, :2]   # (B,T,2)

    # 代理损失：velocity_loss 使用归一化空间 MSE，waypoint_loss 使用物理位置 MSE
    velocity_loss = error.square().mean(dim=(1,2))        # (B,) 对 T,D 求均值
    waypoint_loss = waypoint_error.square().mean(dim=(1,2))  # (B,)
    surrogate = (velocity_loss + omega * waypoint_loss) / scale   # (B,)

    # 前向用 loss_forward，反向用 surrogate 的梯度
    loss = loss_forward.detach() + surrogate - surrogate.detach()
    return loss


# -----------------------------------------------------------------------------
# 统一的 DiffusionNFT 损失入口
# -----------------------------------------------------------------------------

def nft_loss(
    pred_theta: Tensor,
    pred_old: Tensor,
    pred_ref: Tensor,
    target_x_start: Tensor,
    target_waypoints: Optional[Tensor],
    x_t: Tensor,
    t_or_alpha: Tensor,
    sigma: Optional[Tensor] = None,
    advantages: Optional[Tensor] = None,
    beta: float = 0.5,
    beta_kl: float = 0.1,
    adv_clip_max: float = 1.0,
    model_type: str = 'flow',
    prediction_type: str = 'x0',
    loss_type: str = 'original',
    omega: float = 0.01,
    dt: float = 1.0,
    detach_window_size: Optional[int] = None,
    state_mean: Optional[Tensor] = None,
    state_std: Optional[Tensor] = None,
    n_integral: int = 2,
    adaptive_scale: bool = True,
) -> Tuple[Tensor, Dict[str, Tensor]]:
    """
    DiffusionNFT 统一损失函数，支持 Flow 与 VP-Score，支持原始 MSE 或 Hybrid P 范数。

    参数说明与形状：
        pred_theta: (B,T,D) 当前策略预测
        pred_old:   (B,T,D) 旧策略预测，需已 detach
        pred_ref:   (B,T,D) 参考模型预测，需已 detach
        target_x_start: (B,T,D) 干净数据目标（归一化空间）
        target_waypoints: (B,T,2) 绝对位置目标，仅 hybrid 截断梯度时需要
        x_t:        (B,T,D) 加噪状态
        t_or_alpha: (B,) Flow 的时间 t 或 VP 的 alpha
        sigma:      (B,) VP 的 sigma，仅 VP 时必需
        advantages: (B,) 优势值，若 None 则 r=0.5
        beta, beta_kl, adv_clip_max: 标量超参数
        model_type: 'flow' 或 'vp'
        prediction_type: VP 预测类型（'x0','noise','score','v'）
        loss_type: 'original'（MSE）或 'hybrid'（P 范数）
        omega, dt, detach_window_size: Hybrid 参数
        state_mean/std: (D,) 归一化参数（用于 hybrid 积分反归一化）
        n_integral: 参与积分的维度数（通常 2）
        adaptive_scale: 是否对 hybrid/原始损失使用自适应分母

    返回：
        total_loss: 标量
        info: dict，包含各项损失的标量值（已 detach）
    """
    B = pred_theta.shape[0]
    device = pred_theta.device

    # 1. 奖励概率
    if advantages is None:
        r = torch.full((B,), 0.5, device=device)
    else:
        r = _reward_probability(advantages, adv_clip_max)  # (B,)

    # 2. 隐式正负策略
    pos_pred = beta * pred_theta + (1.0 - beta) * pred_old.detach()      # (B,T,D)
    neg_pred = (1.0 + beta) * pred_old.detach() - beta * pred_theta      # (B,T,D)

    # 3. 转换到 x_start
    if model_type == 'flow':
        pos_x_start = flow_prediction_to_x1(pos_pred, x_t, t_or_alpha)  # (B,T,D)
        neg_x_start = flow_prediction_to_x1(neg_pred, x_t, t_or_alpha)
    elif model_type == 'vp':
        if sigma is None:
            raise ValueError("sigma required for vp model")
        pos_x_start = vp_prediction_to_x1(pos_pred, x_t, t_or_alpha, sigma, prediction_type)
        neg_x_start = vp_prediction_to_x1(neg_pred, x_t, t_or_alpha, sigma, prediction_type)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    # 4. 分支损失
    if loss_type == 'original':
        if adaptive_scale:
            def mse_adaptive(pred, target):
                # pred, target: (B,T,D)
                dims = tuple(range(1, pred.ndim))  # 对除 batch 外的所有维度求均值
                with torch.no_grad():
                    scale = (pred - target).abs().mean(dim=dims, keepdim=True).clamp_min(1e-5)
                # scale 形状 (B,1,1) 广播
                return ((pred - target).square() / scale).mean(dim=dims)  # (B,)
            l_pos = mse_adaptive(pos_x_start, target_x_start)
            l_neg = mse_adaptive(neg_x_start, target_x_start)
        else:
            dims = tuple(range(1, pred_theta.ndim))
            l_pos = (pos_x_start - target_x_start).square().mean(dim=dims)  # (B,)
            l_neg = (neg_x_start - target_x_start).square().mean(dim=dims)
    elif loss_type == 'hybrid':
        if state_std is None:
            std_tensor = torch.ones(pred_theta.shape[-1], device=device)
        else:
            std_tensor = state_std.to(device)
        if target_waypoints is None:
            raise ValueError("target_waypoints required for hybrid loss")
        l_pos = hybrid_loss_per_sample(
            pos_x_start, target_x_start, target_waypoints,
            std_tensor, n_integral, omega, dt, detach_window_size, adaptive_scale)  # (B,)
        l_neg = hybrid_loss_per_sample(
            neg_x_start, target_x_start, target_waypoints,
            std_tensor, n_integral, omega, dt, detach_window_size, adaptive_scale)  # (B,)
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    # 5. 策略损失
    # r: (B,), l_pos/l_neg: (B,)，逐元素相乘后取均值，再乘以缩放因子
    policy_loss = (r * l_pos + (1.0 - r) * l_neg).mean() * adv_clip_max / beta

    # 6. KL 正则化（对所有权重求平均）
    kl_loss = (pred_theta - pred_ref.detach()).square().mean()

    total_loss = policy_loss + beta_kl * kl_loss

    # 7. 日志信息
    info = {
        'policy_loss': policy_loss.detach(),
        'kl_loss': kl_loss.detach(),
        'positive_loss': l_pos.mean().detach(),
        'negative_loss': l_neg.mean().detach(),
        'r_mean': r.mean().detach(),
        'total_loss': total_loss.detach(),
    }
    return total_loss, info


# -----------------------------------------------------------------------------
# 便捷包装函数（保持与原代码兼容）
# -----------------------------------------------------------------------------

def nft_loss_original_flow(
    pred_theta, pred_old, pred_ref, target_x1, x_t, t, advantages,
    beta, beta_kl, adv_clip_max
):
    """原始 Flow NFT 损失，使用自适应 MSE。"""
    return nft_loss(
        pred_theta, pred_old, pred_ref, target_x1, None, x_t, t, None,
        advantages, beta, beta_kl, adv_clip_max,
        model_type='flow', prediction_type='x0', loss_type='original',
        adaptive_scale=True
    )

def nft_loss_original_score(
    pred_theta, pred_old, pred_ref, target_x1, x_t, alpha, sigma, advantages,
    beta, beta_kl, adv_clip_max, prediction_type='x0'
):
    """原始 VP/Score NFT 损失，使用自适应 MSE。"""
    return nft_loss(
        pred_theta, pred_old, pred_ref, target_x1, None, x_t, alpha, sigma,
        advantages, beta, beta_kl, adv_clip_max,
        model_type='vp', prediction_type=prediction_type, loss_type='original',
        adaptive_scale=True
    )

def nft_loss_hybrid_flow(
    pred_theta, pred_old, pred_ref, target_x_start, target_waypoints, x_t, t, advantages,
    beta, beta_kl, adv_clip_max, omega=0.01, dt=1.0,
    detach_window_size=None, state_mean=None, state_std=None, n_integral=2
):
    """Flow NFT + Hybrid 损失。"""
    return nft_loss(
        pred_theta, pred_old, pred_ref, target_x_start, target_waypoints, x_t, t, None,
        advantages, beta, beta_kl, adv_clip_max,
        model_type='flow', prediction_type='x0', loss_type='hybrid',
        omega=omega, dt=dt, detach_window_size=detach_window_size,
        state_mean=state_mean, state_std=state_std, n_integral=n_integral,
        adaptive_scale=True
    )

def nft_loss_hybrid_score(
    pred_theta, pred_old, pred_ref, target_x_start, target_waypoints, x_t, alpha, sigma, advantages,
    beta, beta_kl, adv_clip_max, prediction_type='x0', omega=0.01, dt=1.0,
    detach_window_size=None, state_mean=None, state_std=None, n_integral=2
):
    """VP/Score NFT + Hybrid 损失。"""
    return nft_loss(
        pred_theta, pred_old, pred_ref, target_x_start, target_waypoints, x_t, alpha, sigma,
        advantages, beta, beta_kl, adv_clip_max,
        model_type='vp', prediction_type=prediction_type, loss_type='hybrid',
        omega=omega, dt=dt, detach_window_size=detach_window_size,
        state_mean=state_mean, state_std=state_std, n_integral=n_integral,
        adaptive_scale=True
    )


In [ ]:

# -----------------------------------------------------------------------------
# 测试用例
# -----------------------------------------------------------------------------

torch.manual_seed(42)

# 公共参数
B, T, D = 2, 5, 4      # batch size, time steps, feature dims (dx,dy,cos,sin)
device = torch.device('cpu')

# 生成模拟数据
pred_theta = torch.randn(B, T, D, device=device, requires_grad=True)
pred_old = torch.randn_like(pred_theta).detach()
pred_ref = torch.randn_like(pred_theta).detach()

# 干净数据目标（归一化空间的速度/位移增量）
target_x_start = torch.randn(B, T, D, device=device)
# 对应的绝对位置目标（用于 hybrid loss 的 waypoint 项）
# 这里简单用 cumsum 模拟位置真值，实际中应从原始数据转换
target_waypoints = torch.cumsum(target_x_start[:, :, :2], dim=1)  # (B,T,2)

advantages = torch.tensor([0.8, 0.2], device=device)  # 不同奖励

# 归一化参数
state_mean = torch.zeros(D, device=device)
state_std = torch.ones(D, device=device) * 0.5  # 简化：前两维0.5，后两维1，这里全部0.5

# ---------- Flow 模型测试 ----------
t = torch.rand(B, device=device)  # 时间步 (0,1)
x_t_flow = flow_add_noise(target_x_start, t)  # 加噪

# 原始 Flow NFT loss
loss, info = nft_loss_original_flow(
    pred_theta, pred_old, pred_ref, target_x_start, x_t_flow, t, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0
)
print("Flow Original NFT Loss:", loss.item())
print("Info:", {k: round(v.item(), 4) for k, v in info.items()})

# Hybrid Flow NFT loss（使用截断梯度）
loss_h, info_h = nft_loss_hybrid_flow(
    pred_theta, pred_old, pred_ref, target_x_start, target_waypoints,
    x_t_flow, t, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    omega=0.01, dt=0.1, detach_window_size=3,
    state_mean=state_mean, state_std=state_std
)
print("\nFlow Hybrid NFT Loss:", loss_h.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_h.items()})

# ---------- VP-Score 模型测试 ----------
alpha = torch.rand(B, device=device) * 0.5 + 0.5  # 模拟 alpha_t
sigma = torch.sqrt(1 - alpha**2)                  # VP 关系
x_t_vp = vp_add_noise(target_x_start, alpha, sigma)

# 原始 VP NFT loss（预测噪声）
loss_vp, info_vp = nft_loss_original_score(
    pred_theta, pred_old, pred_ref, target_x_start, x_t_vp, alpha, sigma, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0, prediction_type='noise'
)
print("\nVP Original NFT Loss (noise):", loss_vp.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_vp.items()})

# Hybrid VP NFT loss（预测 x0，截断梯度）
loss_vph, info_vph = nft_loss_hybrid_score(
    pred_theta, pred_old, pred_ref, target_x_start, target_waypoints,
    x_t_vp, alpha, sigma, advantages,
    beta=0.5, beta_kl=0.1, adv_clip_max=1.0,
    prediction_type='x0', omega=0.01, dt=0.1, detach_window_size=2,
    state_mean=state_mean, state_std=state_std
)
print("\nVP Hybrid NFT Loss (x0):", loss_vph.item())
print("Info:", {k: round(v.item(), 4) for k, v in info_vph.items()})

# 检查梯度流
print("\nGradient flow test:")
loss_h.backward()
print("pred_theta.grad norm:", pred_theta.grad.norm().item())